# 20 · Métricas de recuperación

Evaluación **asistida y exploratoria** de BM25, 11 densos y 11 híbridos.

In [1]:
from pathlib import Path
from itertools import product
from datetime import datetime, timezone
import hashlib
import json
import math
import platform
import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", None)

ARCHIVOS = {
    'anotaciones': 'pertinencia_completa_N01_N12.json',
    'bm25': 'bm25_resultados_v1.json',
    'densa': 'densa_resultados_v1.json',
    'hibrida': 'hibrida_rrf_resultados_v1.json',
}
K = (5, 10, 20)
SALIDA = 'evaluacion_recuperacion_v1.json'
VERSION = 'EVALUACION_ASISTIDA_V1'

def exigir(condicion, mensaje):
    if not condicion:
        raise ValueError(mensaje)

def firma(obj):
    return hashlib.sha256(json.dumps(obj, ensure_ascii=False, sort_keys=True,
        separators=(',', ':'), allow_nan=False).encode('utf-8')).hexdigest()

carpetas = [p / s for p in (Path.cwd(), *Path.cwd().parents)
            for s in ('.', '07_RAG/Pruebas_RAG', 'Pruebas_RAG')]
CARPETA = next((p for p in carpetas if all((p / n).is_file() for n in ARCHIVOS.values())), None)
if CARPETA is None:
    raise FileNotFoundError('Coloca los cuatro JSON de ARCHIVOS en 07_RAG/Pruebas_RAG/.')
entradas = {k: (CARPETA / n).read_bytes() for k, n in ARCHIVOS.items()}
HUELLAS = {k: hashlib.sha256(v).hexdigest() for k, v in entradas.items()}
A, B, D, H = [json.loads(entradas[k]) for k in ARCHIVOS]
base = A['base']
exigir(firma(base) == A['firma_conjunto'], 'La base de pertinencia cambió.')
control = base['control_no_mostrar_al_evaluador']
for k in ('bm25', 'densa', 'hibrida'):
    exigir(HUELLAS[k] == control['archivos_entrada'][k]['sha256'],
           f'{k}: no es la ejecución con la que se construyó el conjunto de pertinencia.')

casos = {c['caso_id']: c for c in base['casos']}
exigir(len(casos) == len(base['casos']) == 1495, 'Cantidad de casos inesperada.')
exigir(set(casos) == set(A['anotaciones']), 'Faltan o sobran anotaciones.')
QRELS = {n: {} for n in base['necesidades']}
fragmentos = 0
for cid, caso in casos.items():
    j = A['anotaciones'][cid]['asistida']
    estado, grado = j['estado'], j['grado']
    exigir(estado in ('anotada', 'informacion_insuficiente'),
           f'{cid}: pendiente. Carga el JSON completo N01–N12, no el de 502 casos.')
    exigir((estado == 'anotada' and type(grado) is int and grado in (0, 1, 2))
           or (estado == 'informacion_insuficiente' and grado is None), f'{cid}: grado inválido.')
    exigir(j['justificacion'].strip() and j['evaluador'] and j['evidencia_literal'],
           f'{cid}: juicio sin trazabilidad.')
    doc = base['documentos'][caso['article_id']]
    for e in j['evidencia_literal']:
        exigir(e['campo'] in ('titulo', 'abstract'), f'{cid}: campo de evidencia inválido.')
        t = doc[e['campo']]
        exigir(type(e['inicio']) is int and type(e['fin']) is int
               and 0 <= e['inicio'] < e['fin'] <= len(t)
               and t[e['inicio']:e['fin']] == e['texto'], f'{cid}: cita no literal.')
        fragmentos += 1
    nid, aid = caso['necesidad_id'], caso['article_id']
    exigir(aid not in QRELS[nid], f'{cid}: par repetido.')
    QRELS[nid][aid] = grado

CONSULTAS = {q['query_id']: q for q in D['consultas']}
exigir(len(CONSULTAS) == len(D['consultas']) == 24, 'Se esperan 24 consultas.')
exigir(D['consultas'] == H['consultas'], 'Consultas densas e híbridas diferentes.')
for q in CONSULTAS.values():
    n = base['necesidades'][q['necesidad_id']]
    exigir(q['consulta'] == n['consultas'][q['idioma']] and q['division'] == n['division'],
           'El texto o la división de una consulta no coincide.')
for n in QRELS:
    qs = [q for q in CONSULTAS.values() if q['necesidad_id'] == n]
    exigir(len(qs) == 2 and {q['idioma'] for q in qs} == {'es', 'en'}, f'{n}: pareja ES/EN inválida.')
    exigir(any(g == 2 for g in QRELS[n].values()), f'{n}: falta referencia pertinente directa.')

SISTEMAS = {'bm25': B, **D['sistemas'], **H['sistemas']}
exigir(len(D['sistemas']) == len(H['sistemas']) == 11 and len(SISTEMAS) == 23,
       'Se esperan BM25, 11 densos y 11 híbridos.')
exigir(set(SISTEMAS) == set(control['sistemas_incluidos']), 'Sistemas distintos a los del pool.')
RANKINGS = {}
universo = {x['article_id'] for x in B['resultados'][0]['documentos']}
exigir(len(universo) == 519, 'El catálogo debe contener 519 identificadores.')
for sid, sistema in SISTEMAS.items():
    exigir(firma(sistema['resultados']) == sistema['sha256_resultados'], f'{sid}: resultados alterados.')
    filas = {r['query_id']: r for r in sistema['resultados']}
    exigir(len(filas) == 24 and set(filas) == set(CONSULTAS), f'{sid}: consultas incompletas.')
    campo = 'score_bm25' if sid == 'bm25' else ('score_rrf' if sid in H['sistemas'] else 'score_coseno')
    for qid, r in filas.items():
        docs = r['documentos']
        exigir(len(docs) == 519 and {d['article_id'] for d in docs} == universo, f'{sid}/{qid}: IDs distintos.')
        exigir(all(math.isfinite(d[campo]) for d in docs), f'{sid}/{qid}: puntuación no finita.')
        if sid == 'bm25':
            exigir(all(d[campo] >= 0 and ((d['rank'] is None) == (d[campo] == 0)) for d in docs),
                   f'{qid}: tratamiento incorrecto de ceros BM25.')
        else:
            exigir(all(type(d['rank']) is int for d in docs), f'{sid}/{qid}: rango ausente.')
        activos = sorted((d for d in docs if d['rank'] is not None), key=lambda d: d['rank'])
        exigir([d['rank'] for d in activos] == list(range(1, len(activos) + 1)), 'Rangos no consecutivos.')
        exigir(activos == sorted(activos, key=lambda d: (-d[campo], int(d['article_id']))), 'Orden inconsistente.')
        ids = [d['article_id'] for d in activos]
        nid = CONSULTAS[qid]['necesidad_id']
        exigir(set(ids[:max(K)]).issubset(QRELS[nid]), f'{sid}/{qid}: hay documentos sin revisar en top20.')
        RANKINGS[sid, qid] = ids

JUICIOS_HUMANOS = sum(j['humana']['estado'] == 'anotada' for j in A['anotaciones'].values())
pendientes = [{'necesidad_id': n, 'article_id': a} for n, qs in QRELS.items() for a, g in qs.items() if g is None]
resumen_anotaciones = pd.DataFrame([{'necesidad': n, 'division': base['necesidades'][n]['division'],
    'casos': len(q), 'grado_2': sum(g == 2 for g in q.values()), 'grado_1': sum(g == 1 for g in q.values()),
    'grado_0': sum(g == 0 for g in q.values()), 'sin_grado': sum(g is None for g in q.values())}
    for n, q in QRELS.items()])
print('Archivo de juicios:', (CARPETA / ARCHIVOS['anotaciones']).resolve())
print(f'{len(casos)} examinados | {len(casos)-len(pendientes)} con grado | {len(pendientes)} sin grado | 0 sin revisar')
print(f'23 sistemas | 24 consultas | top20 incluido en el conjunto | {fragmentos} citas verificadas')
print('Juicios humanos con grado:', JUICIOS_HUMANOS)
display(resumen_anotaciones)


Archivo de juicios: C:\Users\hazar\Documents\GitHub\Tesis_Multimodelo\07_RAG\Pruebas_RAG\pertinencia_completa_N01_N12.json
1495 examinados | 1488 con grado | 7 sin grado | 0 sin revisar
23 sistemas | 24 consultas | top20 incluido en el conjunto | 2961 citas verificadas
Juicios humanos con grado: 0


,necesidad,division,casos,grado_2,grado_1,grado_0,sin_grado
0,N01,desarrollo,129,1,3,125,0
1,N02,desarrollo,107,2,0,105,0
2,N03,desarrollo,153,1,0,152,0
3,N04,desarrollo,113,1,3,105,4
4,N05,comparacion,121,1,3,117,0
5,N06,comparacion,132,1,0,131,0
6,N07,comparacion,101,1,5,92,3
7,N08,comparacion,127,1,0,126,0
8,N09,comparacion,142,1,0,141,0
9,N10,comparacion,135,1,0,134,0


## 2. Calcular métricas

Cortes: **5, 10 y 20**. Principal: **nDCG@10**, con ganancia $2^g-1$ y descuento $1/\log_2(r+1)$. Se añade nDCG con ganancia lineal como sensibilidad. Precision, Recall y RR consideran pertinentes los grados 1 y 2; las columnas `directa`/`directo` usan solo grado 2. MRR es el promedio de RR truncado en cada corte: no se informa MRR sin límite de profundidad.

**Recall y el ideal de nDCG se calculan sobre los relevantes identificados en el conjunto anotado**, no sobre todos los relevantes del catálogo. El top20 de los sistemas está cubierto por casos; siete carecen de grado.

Para esos siete casos se prueban todas las asignaciones 0/1/2, sin guardarlas como etiquetas. Los extremos `inf`/`sup` son rangos de sensibilidad, **no intervalos de confianza**. Primero se promedian ES/EN usando la misma asignación de grados; después las necesidades. Desarrollo y comparación permanecen separados.

La tabla se ordena por el límite inferior de nDCG, solo para facilitar su lectura. Ese orden no prueba superioridad ni generalización.

In [2]:
METRICAS = ('ndcg_juzgados', 'precision', 'recall_juzgados', 'rr',
            'precision_directa', 'recall_directa_juzgados', 'rr_directo', 'ndcg_lineal_juzgados')

def medir(ranking, qrels, k):
    grados = np.array([qrels[a] for a in ranking[:k]], dtype=float)
    ideal = np.array(sorted(qrels.values(), reverse=True)[:k], dtype=float)
    dcg = lambda g: float(np.sum(g / np.log2(np.arange(len(g)) + 2)))
    ndcg = dcg(2**grados - 1) / dcg(2**ideal - 1)
    ndcg_lineal = dcg(grados) / dcg(ideal)
    resultado = [ndcg]
    for umbral in (1, 2):
        posiciones = np.flatnonzero(grados >= umbral) + 1
        relevantes = sum(g >= umbral for g in qrels.values())
        resultado.extend([len(posiciones) / k, len(posiciones) / relevantes,
                          1 / int(posiciones[0]) if len(posiciones) else 0.0])
    return np.array([*resultado, ndcg_lineal], dtype=float)

prueba = medir(['a', 'b', 'c'], {'a': 0, 'b': 2, 'c': 1}, 2)
exigir(np.allclose(prueba[:4], [(3 / math.log2(3)) / (3 + 1 / math.log2(3)),
                              0.5, 0.5, 0.5]), 'Falló el caso de control.')
exigir(np.allclose(medir([], {'a': 2}, 5), 0), 'Falló el ranking vacío.')
exigir(np.isclose(medir(['a'], {'a': 2}, 5)[1], 0.2), 'P@k debe conservar el denominador k.')

ESCENARIOS, IDS_DUDOSOS = {}, {}
for nid, qrels in QRELS.items():
    dudosos = sorted((a for a, g in qrels.items() if g is None), key=int)
    exigir(len(dudosos) <= 7, f'{nid}: demasiadas combinaciones para este protocolo.')
    IDS_DUDOSOS[nid] = dudosos
    ESCENARIOS[nid] = [{**qrels, **dict(zip(dudosos, grados))}
                      for grados in product((0, 1, 2), repeat=len(dudosos))]

VALORES, filas_consulta = {}, []
for sid in SISTEMAS:
    for qid, q in CONSULTAS.items():
        nid, ranking = q['necesidad_id'], RANKINGS[sid, qid]
        for k in K:
            matriz = np.stack([medir(ranking, juicios, k) for juicios in ESCENARIOS[nid]])
            exigir(np.isfinite(matriz).all() and (matriz >= -1e-12).all()
                   and (matriz <= 1+1e-12).all(), 'Métrica fuera de [0, 1].')
            VALORES[sid, qid, k] = matriz
            nulos = sum(QRELS[nid][a] is None for a in ranking[:k])
            fila = {'sistema': sid, 'query_id': qid, 'necesidad_id': nid,
                    'division': q['division'], 'idioma': q['idioma'], 'k': k,
                    'recuperados_hasta_k': min(k, len(ranking)), 'sin_grado_hasta_k': nulos,
                    'con_grado_hasta_k': min(k, len(ranking))-nulos}
            for i, m in enumerate(METRICAS):
                fila[m+'_inf'], fila[m+'_sup'] = float(matriz[:, i].min()), float(matriz[:, i].max())
            filas_consulta.append(fila)

POR_NECESIDAD, filas_resumen = {}, []
for sid in SISTEMAS:
    for nid in QRELS:
        qs = {q['idioma']: qid for qid, q in CONSULTAS.items() if q['necesidad_id'] == nid}
        for k in K:
            es, en = [VALORES[sid, qs[idioma], k] for idioma in ('es', 'en')]
            for idioma, matriz in (('es', es), ('en', en), ('es_en', (es+en)/2)):
                POR_NECESIDAD[sid, nid, idioma, k] = matriz
    for division in ('desarrollo', 'comparacion'):
        ns = [n for n in QRELS if base['necesidades'][n]['division'] == division]
        for idioma in ('es', 'en', 'es_en'):
            for k in K:
                matrices = [POR_NECESIDAD[sid, n, idioma, k] for n in ns]
                inferior = np.mean([v.min(axis=0) for v in matrices], axis=0)
                superior = np.mean([v.max(axis=0) for v in matrices], axis=0)
                fila = {'sistema': sid, 'division': division, 'idioma': idioma,
                        'k': k, 'n_necesidades': len(ns)}
                for i, m in enumerate(METRICAS):
                    nombre = {'rr': 'mrr', 'rr_directo': 'mrr_directo'}.get(m, m)
                    fila[nombre+'_inf'], fila[nombre+'_sup'] = float(inferior[i]), float(superior[i])
                filas_resumen.append(fila)

resumen = pd.DataFrame(filas_resumen)
comparacion = resumen.query("division == 'comparacion' and idioma == 'es_en' and k == 10").copy()
comparacion = comparacion.sort_values(['ndcg_juzgados_inf', 'ndcg_juzgados_sup', 'sistema'],
                                     ascending=[False, False, True])
print('Métricas calculadas sin modificar etiquetas. Rangos = sensibilidad a los null, no intervalos de confianza.')
print('Comparación: ocho necesidades; primero se promedian sus formulaciones ES/EN.')
display(comparacion[['sistema', 'ndcg_juzgados_inf', 'ndcg_juzgados_sup',
                    'precision_inf', 'precision_sup', 'recall_juzgados_inf',
                    'recall_juzgados_sup', 'mrr_inf', 'mrr_sup']].reset_index(drop=True).round(4))


Métricas calculadas sin modificar etiquetas. Rangos = sensibilidad a los null, no intervalos de confianza.
Comparación: ocho necesidades; primero se promedian sus formulaciones ES/EN.


,sistema,ndcg_juzgados_inf,ndcg_juzgados_sup,precision_inf,precision_sup,recall_juzgados_inf,recall_juzgados_sup,mrr_inf,mrr_sup
0,qwen3_4b,0.9438,0.9742,0.2250,0.2500,0.9196,0.9375,1.0000,1.0000
1,arctic_l_v2,0.9358,0.9649,0.2188,0.2438,0.9107,0.9297,1.0000,1.0000
2,harrier_0_6b,0.9339,0.9672,0.2188,0.2375,0.9062,0.9286,1.0000,1.0000
3,nemotron3_1b,0.9327,0.9762,0.2313,0.2438,0.9219,0.9531,1.0000,1.0000
4,pplx_4b_bf16,0.9325,0.9764,0.2313,0.2438,0.9219,0.9531,1.0000,1.0000
5,qwen3_vl_2b_text,0.9302,0.9745,0.2250,0.2375,0.9062,0.9375,1.0000,1.0000
6,pplx_0_6b,0.9294,0.9636,0.2125,0.2313,0.8984,0.9196,1.0000,1.0000
7,embeddinggemma2_text,0.9224,0.9632,0.2125,0.2250,0.8984,0.9263,1.0000,1.0000
8,qwen3_0_6b,0.9204,0.9642,0.2188,0.2313,0.8906,0.9219,1.0000,1.0000
9,granite_311m_r2,0.9166,0.9588,0.2125,0.2188,0.8889,0.9167,1.0000,1.0000


## 3. Comparar calidad y recursos

Las diferencias entre sistemas usan la **misma asignación** para cada caso dudoso. Se compara nDCG@10 en ocho necesidades de comparación; se muestran por separado los densos, para no confundir el mejor codificador con el mejor híbrido.

Se reutilizan tiempos y picos PyTorch de las ejecuciones guardadas. La latencia resumida es la mediana entre las medianas por consulta de comparación. **Para híbridos solo se midió la fusión**, no su latencia total. No se suman tiempos de ejecuciones distintas ni se interpreta memoria PyTorch como RAM/VRAM total.

In [3]:
IDS_SISTEMAS = sorted(SISTEMAS)
NS_COMPARACION = [n for n in QRELS if base['necesidades'][n]['division'] == 'comparacion']
NS_DUDOSAS = [n for n in NS_COMPARACION if IDS_DUDOSOS[n]]
COMBINACIONES = list(product(*(range(len(ESCENARIOS[n])) for n in NS_DUDOSAS)))
exigir(len(COMBINACIONES) <= 10000, 'Demasiados escenarios globales para esta libreta.')
PUNTAJES = []
for indices in COMBINACIONES:
    eleccion = dict(zip(NS_DUDOSAS, indices))
    PUNTAJES.append([np.mean([POR_NECESIDAD[s, n, 'es_en', 10][eleccion.get(n, 0), 0]
                             for n in NS_COMPARACION]) for s in IDS_SISTEMAS])
PUNTAJES = np.array(PUNTAJES)
filas_pares = []
for i, s1 in enumerate(IDS_SISTEMAS):
    for j in range(i+1, len(IDS_SISTEMAS)):
        s2 = IDS_SISTEMAS[j]
        delta = PUNTAJES[:, i] - PUNTAJES[:, j]
        lo, hi = float(delta.min()), float(delta.max())
        conclusion = ('primero_mayor_en_todos' if lo > 1e-12 else
                      'segundo_mayor_en_todos' if hi < -1e-12 else
                      'iguales_en_todos' if np.allclose(delta, 0, atol=1e-12, rtol=0)
                      else 'no_hay_superioridad_estricta_en_todos')
        filas_pares.append({'primero': s1, 'segundo': s2, 'delta_ndcg10_inf': lo,
                           'delta_ndcg10_sup': hi, 'sensibilidad_anotaciones': conclusion})

sensibilidad_familias = {}
for familia, ids in {'todos': IDS_SISTEMAS, 'densos': list(D['sistemas']),
                     'hibridos': list(H['sistemas'])}.items():
    indices = [IDS_SISTEMAS.index(s) for s in ids]
    valores = PUNTAJES[:, indices]
    posibles = set()
    for fila in valores:
        posibles.update(ids[i] for i, v in enumerate(fila) if abs(v-fila.max()) <= 1e-12)
    sensibilidad_familias[familia] = sorted(posibles)

b_ejec = B['ejecuciones'][-1]
lat_b = {q: float(np.median(v['muestras_ms'])) for q, v in b_ejec['latencias_consulta'].items()}
LATENCIAS, recursos = {}, []
for sid in IDS_SISTEMAS:
    familia = 'lexico' if sid == 'bm25' else ('hibrido' if sid in H['sistemas'] else 'denso')
    modelo_id = H['sistemas'][sid]['codificador'] if familia == 'hibrido' else sid
    ds = D['sistemas'].get(modelo_id)
    ejec = ds['ejecuciones'][-1] if ds else b_ejec
    lat_d = ({v['query_id']: float(np.median(v['total']['muestras_ms']))
              for v in ejec['latencias_consulta']} if ds else {})
    lat_f = ({v['query_id']: float(np.median(v['muestras_ms']))
              for v in H['sistemas'][sid]['latencias_fusion']} if familia == 'hibrido' else {})
    exigir(set(lat_d if ds else lat_b) == set(CONSULTAS), f'{sid}: latencias incompletas.')
    if familia == 'hibrido':
        exigir(set(lat_f) == set(CONSULTAS), f'{sid}: fusión incompleta.')
    latencia = lat_b if familia == 'lexico' else lat_f if familia == 'hibrido' else lat_d
    LATENCIAS[sid] = latencia
    qids = [q for q in CONSULTAS if CONSULTAS[q]['division'] == 'comparacion']
    campos = {'sistema': sid, 'familia': familia, 'modelo': ds['condicion']['modelo'] if ds else 'BM25',
              'dtype_pesos': ds['condicion']['dtype_pesos'] if ds else None,
              'revision_procedencia': ds['condicion'].get('revision_procedencia', 'fijada_en_npz') if ds else None,
              'latencia_mediana_ms_comparacion': float(np.median([latencia[q] for q in qids])),
              'alcance_latencia': 'solo_fusion' if familia == 'hibrido' else 'recuperacion_sin_carga',
              'latencia_hibrida_total_medida_ms': None,
              'carga_modelo_ms': ejec.get('carga_modelo_ms'),
              'pico_pytorch_asignado_bytes': ejec.get('pico_pytorch_asignado_bytes'),
              'pico_pytorch_reservado_bytes': ejec.get('pico_pytorch_reservado_bytes'),
              'alcance_memoria': 'codificador_denso_no_pipeline_hibrido' if familia == 'hibrido' else 'proceso_medido',
              'ram_pico_bytes': ejec.get('ram_pico_bytes'), 'entorno': ejec.get('entorno'),
              'fin_ejecucion_utc': ejec.get('fin_utc'),
              'latencias_mediana_por_consulta_ms': latencia}
    if familia == 'hibrido':
        campos['componentes_no_sumados_ms'] = {'bm25': lat_b, 'densa': lat_d, 'fusion': lat_f}
    recursos.append(campos)

tabla_densa = comparacion[comparacion['sistema'].isin(D['sistemas'])].merge(pd.DataFrame(recursos), on='sistema')
tabla_densa['GPU_PyTorch_GiB'] = tabla_densa['pico_pytorch_asignado_bytes'] / 2**30
print('Recuperación densa: calidad y mediciones existentes; no se crea una puntuación mixta calidad/tiempo.')
display(tabla_densa[['sistema', 'ndcg_juzgados_inf', 'ndcg_juzgados_sup',
                    'latencia_mediana_ms_comparacion', 'GPU_PyTorch_GiB']].round(4))
print('Primeros posibles entre densos al variar solo los null:', sensibilidad_familias['densos'])
print('Los tiempos de los híbridos son solo de fusión; no compararlos como latencia total contra los densos.')


Recuperación densa: calidad y mediciones existentes; no se crea una puntuación mixta calidad/tiempo.


,sistema,ndcg_juzgados_inf,ndcg_juzgados_sup,latencia_mediana_ms_comparacion,GPU_PyTorch_GiB
0,qwen3_4b,0.9438,0.9742,61.8135,7.5605
1,arctic_l_v2,0.9358,0.9649,19.9750,2.1258
2,harrier_0_6b,0.9339,0.9672,46.0828,1.1223
3,nemotron3_1b,0.9327,0.9762,24.1343,2.1367
4,pplx_4b_bf16,0.9325,0.9764,60.2577,7.5572
5,qwen3_vl_2b_text,0.9302,0.9745,48.2532,3.9771
6,pplx_0_6b,0.9294,0.9636,44.6583,2.2319
7,embeddinggemma2_text,0.9224,0.9632,56.7542,0.5181
8,qwen3_0_6b,0.9204,0.9642,45.8885,1.1228
9,granite_311m_r2,0.9166,0.9588,19.7018,0.5985


Primeros posibles entre densos al variar solo los null: ['arctic_l_v2', 'nemotron3_1b', 'pplx_4b_bf16', 'qwen3_4b']
Los tiempos de los híbridos son solo de fusión; no compararlos como latencia total contra los densos.


## 4. Guardar una sola salida

`evaluacion_recuperacion_v1.json` reúne métricas, escenarios, diferencias, recursos, huellas y advertencias. Reejecutar con las mismas entradas y código conserva el resultado; una versión distinta no se sobrescribe.

In [4]:
PROTOCOLO = {
    'tipo_evaluacion': 'asistida_exploratoria_no_validacion_humana',
    'origen_grados': 'anotaciones.asistida; no se mezclan con anotaciones.humana',
    'unidad_documental': 'article_id; sin fusionar registros de texto idéntico',
    'unidad_promedio': 'necesidad; ES/EN se promedian antes de promediar necesidades',
    'principal': 'nDCG_juzgados@10, division comparacion, ES/EN',
    'cortes': list(K), 'relevancia_binaria_principal': 'grado >= 1',
    'relevancia_binaria_adicional': 'grado == 2',
    'ganancia_ndcg_principal': '2**grado - 1',
    'ganancia_ndcg_sensibilidad': 'grado',
    'descuento_dcg': '1/log2(posicion+1), posiciones desde 1',
    'precision': 'relevantes recuperados hasta k / k; posiciones no devueltas no aportan',
    'recall_juzgados': 'relevantes recuperados hasta k / relevantes del conjunto anotado para la necesidad',
    'rr': '1/primera posicion pertinente si está dentro de k; 0 si no; MRR es su promedio',
    'ideal_ndcg': 'orden ideal de los juicios del conjunto por necesidad, no de todo el catálogo',
    'null': 'enumerar grados 0,1,2 sin alterar etiquetas; misma asignación para ES/EN y todos los sistemas',
    'rangos': 'extremos exactos respecto de null, condicionados a los grados asignados; no son IC estadísticos',
    'no_juzgados_fuera_conjunto': 'no se etiquetan ni se incluyen como irrelevantes; cortes máximos top20',
    'empates': 'se respetan rankings existentes, incluida la regla de article_id',
    'tiempos': 'última ejecución registrada; mediana de 10 repeticiones por consulta y mediana entre consultas de comparación',
    'latencia_hibrida': 'solo fusión medida; componentes almacenados separados, sin simular un total',
    'seleccion_definitiva_realizada': False,
}
ADVERTENCIAS = [
    'La libreta anterior guardó salidas con 502 examinados. Esta ejecución exige el JSON completo, sin casos pendientes.',
    'Los juicios son del asistente y no cuentan con validación humana independiente.',
    'Recall y nDCG son relativos a los relevantes identificados en el conjunto; fuera de él pueden faltar relevantes.',
    'Los rangos solo cubren la incertidumbre de los siete null actuales; no errores de las otras etiquetas ni del muestreo.',
    'Las ocho necesidades de comparación son internas y proceden de artículos orientadores; no son 16 tareas independientes.',
    'Cinco necesidades de comparación tienen un solo relevante identificado. Los resultados no prueban generalización.',
    'Se comparan configuraciones ejecutadas, con sus precisiones numéricas y prompts, no arquitecturas en abstracto.',
    'Qwen3-0.6B: revisión inferida de caché; se usó prompt_name=query, pero el prefijo literal no quedó registrado correctamente.',
    'Picos de GPU corresponden al asignador PyTorch; RAM pico no medida. No equivalen a uso total del equipo.',
    'Las repeticiones de latencia no son repeticiones independientes de calidad. No se mide generación ni carga dentro de la búsqueda.',
    'No se modifican anotaciones ni rankings; no se ajustan parámetros según la comparación.',
    'Esta evaluación es temática: no valida justificaciones de pares de autores ni los factores de una predicción de coautoría.',
]
nb_path = CARPETA / '20_evaluacion_recuperacion.ipynb'
exigir(nb_path.is_file(), 'Guarda esta libreta como 20_evaluacion_recuperacion.ipynb en Pruebas_RAG.')
nb = json.loads(nb_path.read_text(encoding='utf-8'))
codigo = '\n\n'.join(''.join(c['source']) for c in nb['cells'] if c['cell_type'] == 'code')
sha_codigo = hashlib.sha256(codigo.encode('utf-8')).hexdigest()
for k, nombre in ARCHIVOS.items():
    exigir(hashlib.sha256((CARPETA/nombre).read_bytes()).hexdigest() == HUELLAS[k], f'{k}: entrada cambió durante la ejecución.')

cuerpo = {
    'version_esquema': VERSION, 'protocolo': PROTOCOLO,
    'fuentes': {k: {'archivo': ARCHIVOS[k], 'sha256': HUELLAS[k]} for k in ARCHIVOS},
    'firma_conjunto': A['firma_conjunto'], 'sha256_codigo': sha_codigo,
    'validacion': {'casos_examinados': len(casos), 'con_grado': len(casos)-len(pendientes),
        'sin_grado': len(pendientes), 'pendientes': 0, 'juicios_humanos_con_grado': JUICIOS_HUMANOS,
        'fragmentos_literales': fragmentos,
        'sistemas': len(SISTEMAS), 'consultas': len(CONSULTAS), 'top20_cubierto_por_casos': True},
    'anotaciones_resumen': resumen_anotaciones.to_dict('records'),
    'casos_informacion_insuficiente': pendientes,
    'metricas_por_consulta': filas_consulta, 'resumen': filas_resumen,
    'diferencias_pareadas_ndcg10_comparacion': filas_pares,
    'primeros_posibles_solo_sensibilidad_null': sensibilidad_familias,
    'escenarios_comparacion_ndcg10': [
        {'grados_solo_para_calculo': {n: {a: ESCENARIOS[n][i][a] for a in IDS_DUDOSOS[n]}
                                    for n, i in zip(NS_DUDOSAS, indices)},
         'puntajes': dict(zip(IDS_SISTEMAS, map(float, fila)))}
        for indices, fila in zip(COMBINACIONES, PUNTAJES)],
    'recursos': recursos, 'advertencias': ADVERTENCIAS,
    'referencias': ['https://ir-measur.es/en/latest/measures.html',
                   'https://www.nist.gov/publications/retrieval-evaluation-incomplete-information'],
}
resultado = {'firma_resultados': firma(cuerpo), 'resultados': cuerpo}
def equivalentes(x, y):
    if isinstance(x, dict) and isinstance(y, dict):
        return x.keys() == y.keys() and all(equivalentes(x[k], y[k]) for k in x)
    if isinstance(x, list) and isinstance(y, list):
        return len(x) == len(y) and all(equivalentes(a, b) for a, b in zip(x, y))
    if type(x) in (int, float) and type(y) in (int, float):
        return math.isclose(x, y, rel_tol=0, abs_tol=1e-12)
    return x == y

destino = CARPETA / SALIDA
if destino.exists():
    anterior = json.loads(destino.read_text(encoding='utf-8'))
    exigir(anterior['firma_resultados'] == firma(anterior['resultados']), 'Salida existente alterada.')
    exigir(equivalentes(anterior['resultados'], cuerpo),
           'Ya existe otra evaluación. Consérvala y cambia SALIDA; no se sobrescribe.')
    print('Resultado equivalente (tolerancia 1e-12): se conserva el archivo existente.')
else:
    temporal = destino.with_suffix('.tmp')
    temporal.write_text(json.dumps(resultado, ensure_ascii=False, indent=2, allow_nan=False), encoding='utf-8')
    temporal.replace(destino)
    print('Guardado:', destino.resolve())
print('Entradas intactas. Sin inferencias nuevas, sin GPU y sin selección definitiva.')


Guardado: C:\Users\hazar\Documents\GitHub\Tesis_Multimodelo\07_RAG\Pruebas_RAG\evaluacion_recuperacion_v1.json
Entradas intactas. Sin inferencias nuevas, sin GPU y sin selección definitiva.
